<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# EarthDaily Agriculture — Extractor Cache Showcase

Demonstrates the per-entity cache layer available on every `BaseExtractor` subclass. The cache stores completed extraction results to a local parquet file so that subsequent runs skip the API for entities already seen.

**Covered in this notebook:**

1. Enabling the cache (via `config` or `setup_*_parameters(use_cache=True)`)
2. Inspecting cache state (`cache_info`, cache file layout)
3. Cold run (full cache miss) → warm run (full cache hit)
4. Partial cache (mix of hits and misses)
5. Parameter-scoped caches (changing setup params creates a new cache file)
6. Per-call `use_cache` override and TTL eviction
7. Cleanup with `clear_cache`

**Extractor used:** `CoverageExtractor` — simplest to run, cache keys on `id + image_id + mask`. The same API applies to every other extractor (each declares its own `cache_key_columns` in `setup_*_parameters()`).

## Step 0: Bootstrap

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

## Step 1: Initialisation

Authenticate against the EarthDaily Agriculture API and prepare a `WorkflowManager`. All extractors inherit the cache configuration from `manager.config`, so any `use_cache` / `cache_dir` / `cache_ttl_days` keys passed here become the defaults.

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager

manager = WorkflowManager("prod", log_to_console=False, log_level="WARNING")

## Step 2: Load entities

Pull a small set of season-fields to keep the demo fast. Any DataFrame with an `id` column and a WKT `geometry` column works — the cache key is based on the entity identifier, not the source.

In [ ]:
manager.load_seasonfields()

print(f"Loaded {len(manager.sfd_list)} entities.")
display(manager.sfd_list[["id", "name"]].head())

## Step 3: How the cache works

The cache layer lives on `BaseExtractor` and is inherited by every extractor. Key pieces:

| Attribute | Purpose |
|---|---|
| `use_cache` (bool) | Master switch — read from `config` or overridden by `setup_*_parameters(use_cache=...)` |
| `cache_dir` (Path) | Folder for parquet cache files (default: `<project>/cache`) |
| `cache_ttl_days` (int) | Rows older than this are treated as stale and evicted on next read |
| `cache_key_columns` (list) | Defined per extractor — dedup key when merging into the cache file |

**Cache filename convention:** `<extractor_name>_<params_hash>_cache.parquet`. Changing any value in `setup_*_parameters()` that is part of the hash produces a different file, so two setups (e.g. NDVI vs EVI) coexist without overlapping.

**Default `cache_key_columns` per extractor** (abbreviated):

| Extractor | Key columns |
|---|---|
| CoverageExtractor | `id`, `image_id`, `mask` |
| VegetationTsExtractor / MRTSExtractor / RegionalExtractor / WeatherExtractor / DiseaseExtractor | `id`, `date` |
| cropidExtractor | `id`, `year` |
| FLMExtractor / ZoningExtractor / DifferenceExtractor | `id` (+ `image_id` for zoning/difference) |
| All processors (Greenness, Emergence, Harvest, Planted, Score, Baresoil, ZARC, InSeasonMonitoring) | `id` |

**Public API used in this notebook:**

- `extractor.cache_info(params=...)` — path, record count, oldest/newest timestamps, stale count
- `extractor.clear_cache(params=...)` — delete the parquet file for that parameter set
- `use_cache=False` passed to a bulk call — temporary bypass without touching the flag
- `extractor.cache_dir`, `extractor.cache_ttl_days` — introspect effective settings

## Step 4: Build a cache-enabled extractor

Two equivalent ways to enable the cache:

1. Pass `use_cache=True` (and optional `cache_dir` / `cache_ttl_days`) in the **config** — inherited by every extractor built from that config.
2. Pass `use_cache=True` directly to **`setup_*_parameters()`** — overrides the config for this extractor only.

We use path (2) below.

In [ ]:
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor

extractor = CoverageExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

extractor.setup_coverage_parameters(
    vegetation_index="NDVI",
    start_date="2025-01-01",
    clear_cover_min=80,
    filter="duplicate",
    mask="auto",
    column_mapping={"crop": "crop.id"},
    use_cache=True,   # <-- enable the cache
)

print(f"use_cache:          {extractor.use_cache}")
print(f"cache_dir:          {extractor.cache_dir}")
print(f"cache_ttl_days:     {extractor.cache_ttl_days}")
print(f"cache_key_columns:  {extractor.cache_key_columns}")

# Clear any cache file from previous runs so the demo starts fresh.
extractor.clear_cache(params=extractor.coverage_params)
print("\nStarting with a cleared cache.")

## Step 5: First run — cold cache (all misses)

No cache file exists yet, so every entity hits the API. Results are written to the cache parquet at the end of the run.

In [ ]:
import time

test_entities = manager.sfd_list.head(20)
print(f"Running on {len(test_entities)} entities (cold cache).")

t0 = time.time()
result_cold = extractor.process_entity_coverage_bulk_parallel(
    entity_list=test_entities,
    max_workers=10,
    skip_export=True,
    prefix="cache_showcase",
    use_cache=True,
)
elapsed_cold = time.time() - t0

print(f"\n--- Cold run ---")
print(f"Elapsed:      {elapsed_cold:.2f}s")
print(f"Cache hits:   {result_cold.get('cache_hit', 'N/A')}")
print(f"Cache misses: {result_cold.get('cache_miss', 'N/A')}")
print(f"Result rows:  {len(result_cold['results_df'])}")

info = extractor.cache_info(params=extractor.coverage_params)
print(f"\nCache file now has {info['records']} records at:\n  {info['path']}")

## Step 6: Second run — warm cache (all hits)

Same entities, same params — every entity should resolve from the parquet cache and no API call is made. Expect the elapsed time to drop by one or two orders of magnitude.

In [ ]:
t0 = time.time()
result_warm = extractor.process_entity_coverage_bulk_parallel(
    entity_list=test_entities,
    max_workers=10,
    skip_export=True,
    prefix="cache_showcase",
    use_cache=True,
)
elapsed_warm = time.time() - t0

print(f"--- Warm run ---")
print(f"Elapsed:      {elapsed_warm:.2f}s")
print(f"Cache hits:   {result_warm.get('cache_hit', 'N/A')}")
print(f"Cache misses: {result_warm.get('cache_miss', 'N/A')}")
print(f"Result rows:  {len(result_warm['results_df'])}")
print(f"\nSpeedup vs cold run: {elapsed_cold / max(elapsed_warm, 0.001):.1f}x")

## Step 7: Partial cache — mix of hits and misses

Extend the entity list so that the first 20 are already cached and the next 10 are new. Only the new entities hit the API.

In [ ]:
extended_entities = manager.sfd_list.head(30)  # 20 cached + 10 new

t0 = time.time()
result_partial = extractor.process_entity_coverage_bulk_parallel(
    entity_list=extended_entities,
    max_workers=10,
    skip_export=True,
    prefix="cache_showcase",
    use_cache=True,
)
elapsed_partial = time.time() - t0

print(f"--- Partial cache run ---")
print(f"Elapsed:      {elapsed_partial:.2f}s")
print(f"Cache hits:   {result_partial.get('cache_hit', 'N/A')}")
print(f"Cache misses: {result_partial.get('cache_miss', 'N/A')}")
print(f"Result rows:  {len(result_partial['results_df'])}")

info = extractor.cache_info(params=extractor.coverage_params)
print(f"\nCache now holds {info['records']} records.")

## Step 8: Data consistency check

Results coming back from the cache must match results from the API. Compare the cold-run and warm-run DataFrames cell by cell.

In [ ]:
import pandas as pd

df_cold = (result_cold["results_df"]
           .drop(columns=["_cached_at"], errors="ignore")
           .sort_values(["id", "image_id"])
           .reset_index(drop=True))
df_warm = (result_warm["results_df"]
           .drop(columns=["_cached_at"], errors="ignore")
           .sort_values(["id", "image_id"])
           .reset_index(drop=True))

print(f"Cold-run shape: {df_cold.shape}")
print(f"Warm-run shape: {df_warm.shape}")
print(f"Shapes match:   {df_cold.shape == df_warm.shape}")

if df_cold.shape == df_warm.shape:
    # NaN-safe cell-by-cell comparison
    equal_mask = df_cold.fillna("__NAN__").eq(df_warm.fillna("__NAN__"))
    mismatches = int((~equal_mask).sum().sum())
    print(f"Cell mismatches: {mismatches}")
    print("Consistency:", "PASSED" if mismatches == 0 else "FAILED")

## Step 9: Inspect the cache file

`cache_info()` returns metadata about the parquet file. Opening the file directly shows the stored rows including the `_cached_at` timestamp used for TTL eviction.

In [ ]:
info = extractor.cache_info(params=extractor.coverage_params)
print("Cache info:")
for k, v in info.items():
    print(f"  {k}: {v}")

cached_df = pd.read_parquet(info["path"])
print(f"\nCached parquet shape: {cached_df.shape}")
print(f"Columns: {list(cached_df.columns)}")
display(cached_df.head(3))

## Step 10: Parameter-scoped caches

The cache filename embeds a hash of the setup parameters. Swap `vegetation_index` from `NDVI` to `EVI` and a **separate** cache file is used — the NDVI file is untouched. This keeps results for different extraction configs isolated.

In [ ]:
extractor_evi = CoverageExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

extractor_evi.setup_coverage_parameters(
    vegetation_index="EVI",        # <-- differs from the NDVI setup above
    start_date="2025-01-01",
    clear_cover_min=80,
    filter="duplicate",
    mask="auto",
    column_mapping={"crop": "crop.id"},
    use_cache=True,
)

ndvi_path = extractor._cache_path(extractor.coverage_params)
evi_path  = extractor_evi._cache_path(extractor_evi.coverage_params)

print(f"NDVI cache path: {ndvi_path.name}")
print(f"EVI  cache path: {evi_path.name}")
print(f"Distinct files:  {ndvi_path != evi_path}")

print("\nNDVI cache records:", extractor.cache_info(params=extractor.coverage_params)["records"])
print("EVI  cache records:", extractor_evi.cache_info(params=extractor_evi.coverage_params)["records"])
print("(EVI cache is empty — running on EVI params will populate it independently.)")

## Step 11: Per-call `use_cache` override

The bulk method accepts `use_cache=False` to bypass the cache for a single call, even when the extractor was built with `use_cache=True`. Useful for forcing a refresh when you suspect upstream data changed.

In [ ]:
# Bypass the cache for this one call — all entities go to the API even though they are cached.
result_bypass = extractor.process_entity_coverage_bulk_parallel(
    entity_list=manager.sfd_list.head(5),
    max_workers=5,
    skip_export=True,
    prefix="cache_bypass",
    use_cache=False,      # <-- override: ignore cache for this call
)

print(f"Result rows:            {len(result_bypass['results_df'])}")
print(f"'cache_hit' key present: {'cache_hit' in result_bypass}  (False when bypassed)")
print(f"Extractor's use_cache:   {extractor.use_cache}  (unchanged)")

## Step 12: TTL behaviour (simulated)

Each cached row carries a `_cached_at` timestamp. On every read, rows older than `cache_ttl_days` are dropped and re-fetched on the next run. To demonstrate without waiting for days to pass, we rewrite a few rows with an artificially old timestamp and ask the cache how many it considers stale.

In [ ]:
cache_path = extractor._cache_path(extractor.coverage_params)
df_cache = pd.read_parquet(cache_path)

# Backdate the first 10 rows to 30 days ago (well beyond the 7-day default TTL).
stale_cutoff = pd.Timestamp.now() - pd.Timedelta(days=30)
df_cache.loc[df_cache.index[:10], "_cached_at"] = stale_cutoff
df_cache.to_parquet(cache_path, index=False)

info = extractor.cache_info(params=extractor.coverage_params)
print(f"Records:        {info['records']}")
print(f"Oldest:         {info.get('oldest')}")
print(f"Newest:         {info.get('newest')}")
print(f"TTL (days):     {extractor.cache_ttl_days}")
print(f"Stale records:  {info.get('stale_records')}  (will be evicted on next cache read)")

## Step 13: Cleanup

`clear_cache(params=...)` deletes the parquet file for a specific parameter set. Leaving the cache in place between runs is usually fine, but when a showcase is done it's polite to clean up.

In [ ]:
extractor.clear_cache(params=extractor.coverage_params)
extractor_evi.clear_cache(params=extractor_evi.coverage_params)

print("NDVI cache:", extractor.cache_info(params=extractor.coverage_params))
print("EVI  cache:", extractor_evi.cache_info(params=extractor_evi.coverage_params))

## Summary

- Enable with `config={"use_cache": True}` **or** `setup_*_parameters(use_cache=True)`.
- Results are stored in `<cache_dir>/<extractor>_<params_hash>_cache.parquet`; different setup params → different file.
- `cache_key_columns` (set in each extractor's `setup_*_parameters()`) defines the dedup key used when merging.
- Second run over the same entities returns from the cache with no API calls; partial runs only fetch new entities.
- `cache_ttl_days` (default 7) evicts old rows automatically on read.
- `use_cache=False` on a bulk call forces a refresh for that call only.
- `cache_info()` and `clear_cache()` provide inspection and cleanup.

The pattern is identical for every other extractor — swap `CoverageExtractor` for `WeatherExtractor`, `VegetationTsExtractor`, `GreennessExtractor`, etc. and the exact same cache API applies.